# Bayan wake v2 — three-phrase model (بيان · يا بيان · Bayan)

**Why v1 failed (measured, not guessed).** v1 detected `يا بيان` at 0.764 but scored
`بيان` 0.0057 and `Bayan` 0.0075 in the browser. Root causes:

1. `target_phrase` was `["bayan","beyan","bayaan"]` — Latin only — and the generator was
   **en-us-libritts-high (an English voice)**, so **Egyptian Arabic بيان was never synthesized**.
2. Synthetic:real was **5000:20 (250:1)** — 20 real clips could not shift the loss.
3. Real clips were phrase-imbalanced by an `i%6` split: train `bayan_ar 6 / bayan_en 7 /
   ya_bayan 7`, held-out `2/1/1`. **بيان had the fewest examples.**
4. `target_recall 0.25` / `target_accuracy 0.6` let auto-training stop at a very weak model.
5. It is **not** a preprocessing bug: the browser chain scored 0.764 on the same
   melspec→embedding→classifier path, proving train/inference feature parity.

**What v2 changes**
- Three **independent, balanced** phrase groups, each synthesized in the right language.
- **Arabic synthetic positives** via NAMAA-Egyptian-TTS (MIT) — used *only as wake training
  data*, unrelated to Bayan's spoken voice.
- **Stratified per-phrase** train/held-out split of the 24 real clips.
- **Phrase-specific adversarial negatives** (بيانات، البيان الصحفي، بيّن، Bayern، banyan…).
- Raised `target_recall`/`target_accuracy` + more steps.
- **Per-phrase evaluation** (held-out real + generated variants + background + non-wake
  speech) whose measured numbers are written into `wake_meta.json`.

Runtime: **GPU (T4)**. Run cells in order; every stage asserts its own output.

In [ ]:
## Setup — pinned, single-source, matched Piper fork, + Arabic TTS for training data
!apt-get -qq update && apt-get -qq install -y espeak-ng
!git clone https://github.com/dscripka/piper-sample-generator
!cd piper-sample-generator && git checkout f1988a4d54eddb23d99e86f0adfef6226a85acc7
!mkdir -p piper-sample-generator/models
!wget -q -O piper-sample-generator/models/en-us-libritts-high.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'
!pip install -q espeak-phonemizer webrtcvad
!git clone https://github.com/dscripka/openwakeword
!cd openwakeword && git checkout 368c03716d1e92591906a84949bc477f3a834455
!pip install -q -e ./openwakeword
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.12.0 acoustics==0.2.6 pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19 onnxscript soundfile
# Arabic TTS used ONLY to synthesize wake-word TRAINING audio (not Bayan's voice).
!pip install -q chatterbox-tts
import os, pathlib
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
import subprocess as _sp
for f in ["embedding_model.onnx","embedding_model.tflite","melspectrogram.onnx","melspectrogram.tflite"]:
    _sp.run(["wget","-q","https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/"+f,
             "-O","./openwakeword/openwakeword/resources/models/"+f], check=True)

# ---- modern-PyTorch compatibility patches (identical to the run that trained v1) ----
_gs = pathlib.Path("piper-sample-generator/generate_samples.py"); _s = _gs.read_text()
if "torch.load(model_path)" in _s and "weights_only" not in _s:
    _gs.write_text(_s.replace("torch.load(model_path)", "torch.load(model_path, weights_only=False)")); print("patched piper -> weights_only=False")
_tp = pathlib.Path("openwakeword/openwakeword/train.py"); _t = _tp.read_text()
if "opset_version=13)" in _t and "dynamo=False" not in _t:
    _tp.write_text(_t.replace("opset_version=13)", "opset_version=13, dynamo=False)")); print("patched train.py -> dynamo=False")
import importlib.util as _ilu
_ta = _ilu.find_spec("torch_audiomentations")
if _ta and _ta.submodule_search_locations:
    _io = pathlib.Path(list(_ta.submodule_search_locations)[0]) / "utils" / "io.py"; _x = _io.read_text()
    if "info = torchaudio.info(str(file_path))" in _x:
        _io.write_text(_x.replace("info = torchaudio.info(str(file_path))",
            'import soundfile as _sf; _si = _sf.info(str(file_path)); info = type("MD", (), {"num_frames": _si.frames, "sample_rate": _si.samplerate})()'))
        print("patched torch_audiomentations -> soundfile.info")
_dp = pathlib.Path("openwakeword/openwakeword/data.py"); _d = _dp.read_text()
if "metadata = torchaudio.info(clip)" in _d:
    _dp.write_text(_d.replace("metadata = torchaudio.info(clip)",
        "import soundfile as _sf; _mi = _sf.info(str(clip)); metadata = type('MD', (), {'num_frames': _mi.frames, 'sample_rate': _mi.samplerate})()"))
    print("patched data.py get_clip_duration -> soundfile")
print("setup complete")

In [ ]:
import os, sys, glob, math, json, shutil, hashlib, subprocess, random
import numpy as np, torch, yaml, datasets, scipy, scipy.io.wavfile as wf
from pathlib import Path
from scipy.signal import resample_poly
from tqdm import tqdm

MODEL_VERSION   = "bayan-wake-v2-threephrase"
DATASET_VERSION = "v2-egyptian-pilot-balanced"
SR = 16000

# The three REQUIRED wake phrases, each with the language its synthesis must use.
PHRASES = [
    {"id": "bayan_ar", "text": "بيان",    "lang": "ar", "clip_tag": "bayan_ar"},
    {"id": "ya_bayan", "text": "يا بيان", "lang": "ar", "clip_tag": "ya_bayan"},
    {"id": "bayan_en", "text": "Bayan",   "lang": "en", "clip_tag": "bayan_en"},
]
# Latin variants only for the ENGLISH phrase (v1's mistake was using these for Arabic too).
EN_VARIANTS = ["bayan", "bayaan", "beyan", "ba yan"]

def to_16k_mono_pcm16(path):
    sr, d = wf.read(path)
    x = d.astype(np.float32) / (32768.0 if d.dtype == np.int16 else 1.0)
    if x.ndim > 1: x = x.mean(axis=1)
    if sr != SR:
        g = math.gcd(int(sr), SR); x = resample_poly(x, SR // g, int(sr) // g)
    wf.write(path, SR, (np.clip(x, -1, 1) * 32767).astype(np.int16))

print("v2 config:", MODEL_VERSION, "|", DATASET_VERSION)

In [ ]:
# Upload wake_data_v1-egyptian-pilot.zip (the 24 real recordings — unchanged).
from google.colab import files
import zipfile
up = files.upload()
z = [f for f in up if f.endswith(".zip")][0]
zipfile.ZipFile(z).extractall(".")
real = sorted(glob.glob("wake_data/positive/**/*.wav", recursive=True))
def tag_of(p):
    b = os.path.basename(p)
    for ph in PHRASES:
        if ph["clip_tag"] in b: return ph["id"]
    return None
from collections import Counter
print("real clips:", len(real), dict(Counter(tag_of(p) for p in real)))
assert len(real) == 24, "expected the 24-clip pilot set"


In [ ]:
# Data: MIT RIRs + background (AudioSet via dataset-server assets + FMA) + precomputed features.
# NOTE: the old AudioSet .tar URL is dead (repo moved to parquet); we pull assets instead.
os.makedirs("mit_rirs", exist_ok=True)
rir = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir, desc="rirs"):
    wf.write(os.path.join("mit_rirs", row["audio"]["path"].split("/")[-1]), SR, (row["audio"]["array"]*32767).astype(np.int16))

import requests, io, librosa, soundfile as _sf
os.makedirs("audioset_16k", exist_ok=True)
API = "https://datasets-server.huggingface.co/rows?dataset=agkphysics/AudioSet&config=balanced&split=train&length=100&offset="
urls = [r["row"]["audio"][0]["src"] for off in (0,100,200) for r in requests.get(API+str(off), timeout=180).json()["rows"]]
for i,u in enumerate(tqdm(urls, desc="audioset")):
    _sf.write(f"audioset_16k/as_{i:04d}.wav", (librosa.load(io.BytesIO(requests.get(u, timeout=180).content), sr=SR, mono=True)[0]*32767).astype("int16"), SR, subtype="PCM_16")

os.makedirs("fma", exist_ok=True)
fma = iter(datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True).cast_column("audio", datasets.Audio(sampling_rate=SR)))
for i in tqdm(range(120), desc="fma"):
    row = next(fma); wf.write(os.path.join("fma", f"fma_{i:04d}.wav"), SR, (row["audio"]["array"]*32767).astype(np.int16))

!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print("rirs", len(glob.glob("mit_rirs/*.wav")), "audioset", len(glob.glob("audioset_16k/*.wav")), "fma", len(glob.glob("fma/*.wav")))
assert len(glob.glob("mit_rirs/*.wav")) and len(glob.glob("audioset_16k/*.wav")) and len(glob.glob("fma/*.wav"))

In [ ]:
# ARABIC synthetic positives — the ingredient v1 was missing entirely.
# NAMAA-Egyptian-TTS (MIT) default voice, varied for acoustic diversity. This audio is
# TRAINING DATA ONLY and has nothing to do with Bayan's spoken voice.
from chatterbox import ChatterboxTTS
import inspect
tts = ChatterboxTTS.from_pretrained("NAMAA-Space/NAMAA-Egyptian-TTS")
sig = set(inspect.signature(tts.generate).parameters)
LANG_KW = "language" if "language" in sig else ("lang" if "lang" in sig else None)
print("generate() params:", sorted(sig))

AR_PER_PHRASE = 1200          # per Arabic phrase; balanced with the English count below
def gen_arabic(text, outdir, n):
    os.makedirs(outdir, exist_ok=True)
    made = len(glob.glob(outdir+"/*.wav"))
    pbar = tqdm(total=n, initial=made, desc=os.path.basename(outdir))
    while made < n:
        kw = {}
        if LANG_KW: kw[LANG_KW] = "ar"
        # vary prosody/temperature for diversity (arg names differ across versions)
        for opt, val in (("temperature", random.choice([0.6,0.8,1.0])),
                         ("exaggeration", random.choice([0.3,0.5,0.7])),
                         ("cfg_weight", random.choice([0.3,0.5]))):
            if opt in sig: kw[opt] = val
        try: w = tts.generate(text, **kw)
        except Exception as e:
            print("generate failed:", e); break
        a = np.asarray(w, dtype=np.float32).squeeze()
        p = os.path.join(outdir, f"ar_{made:05d}.wav")
        wf.write(p, getattr(tts, "sr", 24000), (np.clip(a,-1,1)*32767).astype(np.int16))
        to_16k_mono_pcm16(p)
        made += 1; pbar.update(1)
    pbar.close()
    return len(glob.glob(outdir+"/*.wav"))

counts = {}
for ph in [p for p in PHRASES if p["lang"] == "ar"]:
    counts[ph["id"]] = gen_arabic(ph["text"], f"syn/{ph['id']}", AR_PER_PHRASE)
print("arabic synthetic:", counts)
assert all(v > 0 for v in counts.values()), "Arabic synthesis produced nothing — check the NAMAA generate() signature above"


In [ ]:
# ENGLISH synthetic positives for the "Bayan" phrase (piper fork, English voice — correct here).
os.makedirs("syn/bayan_en", exist_ok=True)
EN_TOTAL = 1200
per = max(1, EN_TOTAL // len(EN_VARIANTS))
for v in EN_VARIANTS:
    d = f"syn/bayan_en_{v.replace(' ','_')}"; os.makedirs(d, exist_ok=True)
    subprocess.run([sys.executable, "piper-sample-generator/generate_samples.py", v,
                    "--model", "piper-sample-generator/models/en-us-libritts-high.pt",
                    "--max-samples", str(per), "--output-dir", d, "--batch-size", "50"],
                   capture_output=True, text=True)
    for p in glob.glob(d+"/*.wav"): to_16k_mono_pcm16(p)
    print(v, "->", len(glob.glob(d+"/*.wav")))
# consolidate english variants into one phrase dir
for p in glob.glob("syn/bayan_en_*/*.wav"):
    shutil.move(p, os.path.join("syn/bayan_en", os.path.basename(os.path.dirname(p))+"_"+os.path.basename(p)))
print("english synthetic:", len(glob.glob("syn/bayan_en/*.wav")))
for ph in PHRASES:
    n = len(glob.glob(f"syn/{ph['id']}/*.wav")); print(f"  {ph['id']}: {n}")
    assert n > 0, f"no synthetic positives for {ph['id']}"

In [ ]:
# Config + STRATIFIED per-phrase split + balanced positive set.
config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)
config["model_name"] = "bayan"
# All three phrases are targets of ONE binary model (openWakeWord activates on any of them).
config["target_phrase"] = ["بيان", "يا بيان", "bayan"]
config["custom_negative_phrases"] = [
    "بيانات", "البيان", "البيان الصحفي", "بيان الحساب", "بيّن", "تبيان", "بيانو",
    "بيانات الشحنة", "أعلن", "بيروت", "banyan", "Bayern", "buy an", "by Ann", "bay window", "biryani",
]
config["n_samples"] = 3600            # 1200 per phrase, balanced (v1: 5000 English-only)
config["n_samples_val"] = 900
config["steps"] = 50000               # v1: 15000
config["target_accuracy"] = 0.85      # v1: 0.60
config["target_recall"] = 0.70        # v1: 0.25
config["target_false_positives_per_hour"] = 0.2
config["background_paths"] = ["./audioset_16k", "./fma"]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
config["augmentation_rounds"] = 2     # more variety from the same clips
yaml.dump(config, open("bayan.yaml","w"), allow_unicode=True)

TRAIN = "openwakeword/openwakeword/train.py"
BASE  = "my_custom_model/bayan"
CLIP_DIRS = [f"{BASE}/{d}" for d in ("positive_train","positive_test","negative_train","negative_test")]
FEATS = [f"{BASE}/{n}.npy" for n in ("positive_features_train","positive_features_test","negative_features_train","negative_features_test")]
for d in CLIP_DIRS: os.makedirs(d, exist_ok=True)

# Stratified split: hold out 2 real clips PER PHRASE (v1 held out 2/1/1 by accident).
random.seed(7)
held = {}
for ph in PHRASES:
    clips = sorted([p for p in real if tag_of(p) == ph["id"]])
    random.shuffle(clips)
    held[ph["id"]] = clips[:2]
    for i, p in enumerate(clips):
        dst = f"{BASE}/positive_test" if p in held[ph["id"]] else f"{BASE}/positive_train"
        out = os.path.join(dst, f"real_{ph['id']}_{i:02d}.wav"); shutil.copy(p, out); to_16k_mono_pcm16(out)
# Balanced synthetic positives into positive_train (equal per phrase).
PER = min(len(glob.glob(f"syn/{p['id']}/*.wav")) for p in PHRASES)
for ph in PHRASES:
    for i, p in enumerate(sorted(glob.glob(f"syn/{ph['id']}/*.wav"))[:PER]):
        shutil.copy(p, f"{BASE}/positive_train/syn_{ph['id']}_{i:05d}.wav")
print("balanced synthetic per phrase:", PER)
print("positive_train:", len(glob.glob(BASE+"/positive_train/*.wav")), "| positive_test:", len(glob.glob(BASE+"/positive_test/*.wav")))
print("held-out per phrase:", {k: [os.path.basename(x) for x in v] for k,v in held.items()})
json.dump({k:[os.path.basename(x) for x in v] for k,v in held.items()}, open("heldout.json","w"))

In [ ]:
# Stage harness: real tracebacks, compat shim for the subprocess, fail-hard.
COMPAT = os.path.abspath("_bayan_compat"); os.makedirs(COMPAT, exist_ok=True)
open(os.path.join(COMPAT, "sitecustomize.py"), "w").write('''
import os
if os.environ.get("BAYAN_COMPAT") == "1":
    try:
        import torch, torchaudio, soundfile as _sf
        _ol = torch.load
        def _load(*a, **k):
            k.setdefault("weights_only", False); return _ol(*a, **k)
        torch.load = _load
        def _info(path, *a, **k):
            i = _sf.info(str(path))
            return type("MD", (), {"num_frames": i.frames, "sample_rate": i.samplerate, "num_channels": i.channels, "bits_per_sample": 16})()
        if not hasattr(torchaudio, "info"): torchaudio.info = _info
    except Exception as e:
        print("compat shim warning:", e)
''')
def run_stage(title, flags):
    env = dict(os.environ); env["PYTHONPATH"] = COMPAT + os.pathsep + env.get("PYTHONPATH",""); env["BAYAN_COMPAT"] = "1"
    print("="*30, title, "="*30)
    p = subprocess.run([sys.executable, TRAIN, "--training_config", "bayan.yaml"] + flags,
                       capture_output=True, text=True, env=env)
    print(p.stdout[-5000:])
    if p.returncode != 0:
        print("---------- REAL STDERR ----------"); print(p.stderr[-12000:])
        raise RuntimeError(f"{title} FAILED (exit {p.returncode})")
    print(f"[{title} completed]")
print("harness ready")

In [ ]:
# STEP 1 — generate/augment features. Our positives already exist, so generation only
# tops up adversarial negatives; then features are computed for everything.
run_stage("STEP 1 generate_clips", ["--generate_clips"])
for d in CLIP_DIRS:
    for p in glob.glob(d+"/*.wav"): to_16k_mono_pcm16(p)
bad = [p for d in CLIP_DIRS for p in glob.glob(d+"/*.wav") if wf.read(p)[0] != SR]
assert not bad, f"not 16 kHz: {bad[:5]}"
print("clips:", {os.path.basename(d): len(glob.glob(d+"/*.wav")) for d in CLIP_DIRS})
run_stage("STEP 2 augment_clips", ["--augment_clips", "--overwrite"])
for f in FEATS:
    assert os.path.exists(f) and os.path.getsize(f) > 0, f"missing {f}"
    print("  ", os.path.basename(f), np.load(f, mmap_mode="r").shape)

In [ ]:
# STEP 3 — train.
assert all(os.path.exists(f) and os.path.getsize(f) > 0 for f in FEATS), "features missing"
before = {p for p in glob.glob("my_custom_model/**/*.onnx", recursive=True)}
run_stage("STEP 3 train_model", ["--train_model"])
new = [p for p in glob.glob("my_custom_model/**/*.onnx", recursive=True) if p not in before] or \
      [p for p in glob.glob("my_custom_model/**/*.onnx", recursive=True)]
assert new, "no classifier ONNX produced"
CLS = max(new, key=os.path.getsize); print("classifier:", CLS, os.path.getsize(CLS), "bytes")

In [ ]:
# EVALUATION — per phrase, multiple samples, plus background and non-wake speech.
# These measured numbers go into wake_meta.json (never invented).
import onnxruntime as ort
from openwakeword.utils import AudioFeatures
sys.path.insert(0, os.path.abspath("openwakeword"))
AF = AudioFeatures(device="cpu")
cls = ort.InferenceSession(CLS, providers=["CPUExecutionProvider"])
IN = cls.get_inputs()[0].name
import time

def score_clip(path):
    sr, d = wf.read(path)
    if d.ndim > 1: d = d[:,0]
    if sr != SR:
        g = math.gcd(int(sr), SR); d = (resample_poly(d.astype(np.float32), SR//g, int(sr)//g)).astype(np.int16)
    x = d.astype(np.int16)
    if len(x) < SR: x = np.pad(x, (0, SR-len(x)))
    best, lat = 0.0, []
    AF.reset() if hasattr(AF, "reset") else None
    step = 1280
    feats = []
    for i in range(0, len(x)-step+1, step):
        t0 = time.time()
        AF._streaming_features(x[i:i+step])
        f = AF.get_features(16)
        if f.shape[1] == 16:
            s = float(cls.run(None, {IN: f.astype(np.float32)})[0].reshape(-1)[0]); best = max(best, s)
        lat.append((time.time()-t0)*1000)
    return best, (float(np.mean(lat)) if lat else float("nan"))

THRESH = 0.5
results = {}
for ph in PHRASES:
    # held-out real clips (never trained on) + freshly generated variants
    hold = [p for p in real if os.path.basename(p) in held[ph["id"]]]
    variants = sorted(glob.glob(f"syn/{ph['id']}/*.wav"))[-15:]   # last 15 = unused tail
    sc_h = [score_clip(p)[0] for p in hold]
    sc_v = [score_clip(p)[0] for p in variants]
    allsc = sc_h + sc_v
    det = sum(1 for s in allsc if s >= THRESH)
    results[ph["id"]] = {"phrase": ph["text"], "heldout_real_scores": [round(s,4) for s in sc_h],
                         "variant_scores_max": round(max(sc_v),4) if sc_v else None,
                         "detected": det, "of": len(allsc),
                         "detectionRate": round(det/max(1,len(allsc)), 3)}
    print(ph["id"], results[ph["id"]])

# False activations: background + non-wake speech (adversarial negatives from training pool)
neg_pool = (sorted(glob.glob("audioset_16k/*.wav"))[:40] + sorted(glob.glob("fma/*.wav"))[:20]
            + sorted(glob.glob(f"{BASE}/negative_test/*.wav"))[:60])
fa, lats, hours = 0, [], 0.0
for p in tqdm(neg_pool, desc="false-activation"):
    s, l = score_clip(p); lats.append(l)
    sr_, d_ = wf.read(p); hours += len(d_)/sr_/3600
    if s >= THRESH: fa += 1
fa_per_hour = fa / hours if hours > 0 else None
print(f"false activations: {fa} over {hours*60:.1f} min -> {fa_per_hour:.2f}/hour" if fa_per_hour is not None else "n/a")
METRICS = {"thresholdUsed": THRESH, "perPhrase": results,
           "falseActivations": {"count": fa, "audioMinutes": round(hours*60,1),
                                "perHour": round(fa_per_hour,3) if fa_per_hour is not None else None},
           "meanInferenceMsPerFrame": round(float(np.mean(lats)),2) if lats else None}
print(json.dumps(METRICS, ensure_ascii=False, indent=1))

In [ ]:
# EXPORT v2 — only if every phrase meets the bar. Writes MEASURED metrics into wake_meta.json.
MIN_RATE = 0.6   # each phrase must be detected in >=60% of its held-out+variant samples
weak = {k: v["detectionRate"] for k, v in results.items() if v["detectionRate"] < MIN_RATE}
print("weak phrases:", weak if weak else "none")

out = "bayan-wake-out"; os.makedirs(out, exist_ok=True)
shutil.copy(CLS, f"{out}/bayan_wake.onnx")
res = "openwakeword/openwakeword/resources/models"
for m in ("melspectrogram.onnx", "embedding_model.onnx"): shutil.copy(f"{res}/{m}", f"{out}/{m}")
sha = hashlib.sha256(open(f"{out}/bayan_wake.onnx","rb").read()).hexdigest()
meta = {
  "modelVersion": MODEL_VERSION, "datasetVersion": DATASET_VERSION,
  "trainer": "openWakeWord automatic training (v2: three balanced phrases, Arabic synthetic positives)",
  "wakePhrases": [p["text"] for p in PHRASES],
  "targetPhrase": config["target_phrase"], "customNegativePhrases": config["custom_negative_phrases"],
  "sampleRate": SR, "inferenceChain": ["melspectrogram.onnx","embedding_model.onnx","bayan_wake.onnx"],
  "threshold": THRESH, "consecutiveFrames": 2, "cooldownMs": 2000,
  "onnxSha256": sha,
  "realPositives": len(real), "syntheticPerPhrase": PER, "steps": config["steps"],
  "targetRecall": config["target_recall"], "targetAccuracy": config["target_accuracy"],
  "heldOutPerPhrase": {k: v for k, v in json.load(open("heldout.json")).items()},
  "metrics": METRICS,
  "acceptance": {"minDetectionRatePerPhrase": MIN_RATE, "weakPhrases": weak,
                 "passed": len(weak) == 0},
}
open(f"{out}/wake_meta.json","w").write(json.dumps(meta, ensure_ascii=False, indent=2))
print(json.dumps(meta, ensure_ascii=False, indent=2)[:1500])
zipname = f"bayan-wake-model_{MODEL_VERSION}"
shutil.make_archive(zipname, "zip", out)
print("\nsha256:", sha)
print("ZIP:", zipname + ".zip")
from google.colab import files as F; F.download(zipname + ".zip")
if weak: print("\n*** DO NOT SHIP: these phrases are below the bar:", weak, "— retrain with more data for them.")
else:    print("\nAll three phrases meet the acceptance bar — safe to validate in the web app.")